In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print(torch.cuda.get_device_properties(0).total_memory/1024**3)

In [ ]:
%pip install -q \
    "vllm==0.19.0" \
    "pyngrok>=7,<8"

In [ ]:
import torch
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
VLLM_API_KEY = userdata.get("VLLM_API_KEY")

assert NGROK_AUTHTOKEN, "Thiếu NGROK_AUTHTOKEN"
assert VLLM_API_KEY, "Thiếu VLLM_API_KEY"
assert torch.cuda.is_available(), "Không tìm thấy CUDA"
assert torch.cuda.is_bf16_supported(), "GPU không hỗ trợ BF16"

gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3

print(f"GPU: {gpu.name}")
print(f"VRAM: {vram_gib:.1f} GiB")
print(f"CUDA: {torch.version.cuda}")

print(NGROK_AUTHTOKEN)

if vram_gib < 20:
    raise RuntimeError("Cần GPU khoảng 20 GiB trở lên; hãy lấy L4/A100.")

In [ ]:
import os
import json
import torch
from pathlib import Path

gpu = torch.cuda.get_device_properties(0)
gpu_name = gpu.name
gpu_memory_gib = gpu.total_memory / 1024**3
cpu_count = os.cpu_count() or 1

print(
    f"GPU={gpu_name}, "
    f"VRAM={gpu_memory_gib:.1f} GiB, "
    f"CPU={cpu_count}"
)

# ============================================================
# vLLM + Chandra configuration based on GPU
# ============================================================

if ("H100" in gpu_name) or (gpu_memory_gib >= 90):
    # H100 80GB / 94GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "256"
    VLLM_MAX_BATCHED_TOKENS = "65536"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    CHANDRA_MAX_WORKERS = 256
    CHANDRA_RENDER_PROCESSES = min(32, cpu_count)

elif "A100" in gpu_name and gpu_memory_gib >= 70:
    # A100 80GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "128"
    VLLM_MAX_BATCHED_TOKENS = "32768"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    CHANDRA_MAX_WORKERS = 128
    CHANDRA_RENDER_PROCESSES = min(24, cpu_count)

elif "A100" in gpu_name:
    # A100 40GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "64"
    VLLM_MAX_BATCHED_TOKENS = "16384"
    VLLM_GPU_MEMORY_UTILIZATION = "0.95"

    CHANDRA_MAX_WORKERS = 64
    CHANDRA_RENDER_PROCESSES = min(16, cpu_count)

elif "L4" in gpu_name:
    # NVIDIA L4 24GB
    VLLM_DTYPE = "bfloat16"
    VLLM_MAX_NUM_SEQS = "32"
    VLLM_MAX_BATCHED_TOKENS = "8192"
    VLLM_GPU_MEMORY_UTILIZATION = "0.92"

    CHANDRA_MAX_WORKERS = 32
    CHANDRA_RENDER_PROCESSES = min(8, cpu_count)

elif "T4" in gpu_name:
    # NVIDIA T4 16GB
    VLLM_DTYPE = "float16"
    VLLM_MAX_NUM_SEQS = "8"
    VLLM_MAX_BATCHED_TOKENS = "4096"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    CHANDRA_MAX_WORKERS = 8
    CHANDRA_RENDER_PROCESSES = min(4, cpu_count)

else:
    # Safe fallback
    VLLM_DTYPE = "auto"
    VLLM_MAX_NUM_SEQS = "8"
    VLLM_MAX_BATCHED_TOKENS = "4096"
    VLLM_GPU_MEMORY_UTILIZATION = "0.90"

    CHANDRA_MAX_WORKERS = 8
    CHANDRA_RENDER_PROCESSES = min(4, cpu_count)


# ============================================================
# Export vLLM config for %%bash
# ============================================================

os.environ["VLLM_DTYPE"] = VLLM_DTYPE
os.environ["VLLM_MAX_NUM_SEQS"] = VLLM_MAX_NUM_SEQS
os.environ["VLLM_MAX_BATCHED_TOKENS"] = VLLM_MAX_BATCHED_TOKENS
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = (
    VLLM_GPU_MEMORY_UTILIZATION
)


# ============================================================
# Print detected profile
# ============================================================

print("\nvLLM config:")
print(f"  dtype={VLLM_DTYPE}")
print(f"  max_num_seqs={VLLM_MAX_NUM_SEQS}")
print(f"  max_num_batched_tokens={VLLM_MAX_BATCHED_TOKENS}")
print(
    f"  gpu_memory_utilization="
    f"{VLLM_GPU_MEMORY_UTILIZATION}"
)

print("\nChandra config:")
print(f"  max_workers={CHANDRA_MAX_WORKERS}")
print(f"  render_processes={CHANDRA_RENDER_PROCESSES}")

In [ ]:
%%bash

echo "Starting Chandra vLLM with:"
echo "  dtype=$VLLM_DTYPE"
echo "  max_num_seqs=$VLLM_MAX_NUM_SEQS"
echo "  max_num_batched_tokens=$VLLM_MAX_BATCHED_TOKENS"
echo "  gpu_memory_utilization=$VLLM_GPU_MEMORY_UTILIZATION"

nohup vllm serve datalab-to/chandra-ocr-2 \
    --host 0.0.0.0 \
    --port 8000 \
    --served-model-name chandra \
    --dtype "$VLLM_DTYPE" \
    --max-model-len 14000 \
    --max-num-seqs "$VLLM_MAX_NUM_SEQS" \
    --max-num-batched-tokens "$VLLM_MAX_BATCHED_TOKENS" \
    --gpu-memory-utilization "$VLLM_GPU_MEMORY_UTILIZATION" \
    --limit-mm-per-prompt '{"image":1}' \
    --mm-processor-kwargs '{"min_pixels":3136,"max_pixels":6291456}' \
    --enable-chunked-prefill \
    --enable-prefix-caching \
    --generation-config vllm \
    > /content/vllm.log 2>&1 &

echo $! > /content/vllm.pid

echo "vLLM PID: $(cat /content/vllm.pid)"
echo "Log: /content/vllm.log"

In [ ]:
!nvidia-smi

In [ ]:
import requests
import time

HEADERS = {
    "Authorization": f"Bearer {VLLM_API_KEY}",
}

deadline = time.time() + 30 * 60
ready = False

while time.time() < deadline:

    try:
        response = requests.get(
            "http://127.0.0.1:8000/v1/models",
            headers=HEADERS,
            timeout=5,
        )

        if response.status_code == 200:
            print(response.json())
            ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(10)

if not ready:
    # In log khi fail
    print("--- vLLM log ---")
    !tail -200 vllm.log
    raise RuntimeError("vLLM chưa sẵn sàng.")

print("vLLM ready.")

In [ ]:
# from pyngrok import ngrok

# ngrok.set_auth_token(NGROK_AUTHTOKEN)

# # Đóng tunnel cũ nếu chạy lại cell.
# ngrok.kill()

# tunnel = ngrok.connect(
#     addr=8000,
#     proto="http",
#     bind_tls=True,
# )

# PUBLIC_URL = tunnel.public_url.rstrip("/")

# print("Public URL:", PUBLIC_URL)
# print("VLLM_API_BASE:", f"{PUBLIC_URL}/v1")


# response = requests.get(
#     f"{PUBLIC_URL}/v1/models",
#     headers=HEADERS,
#     timeout=30,
# )
# print(response.status_code)
# print(response.json())

In [ ]:
%%bash
git clone -b baseline --single-branch \
  https://github.com/iSE-UET-VNU/AXIOM_DE-RD.git \
  /content/AXIOM_DE-RD

In [ ]:
%cd /content/AXIOM_DE-RD
%pip install -q -e ".[chandra2]"

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import json
import os

drive.mount("/content/drive")
REPO_DIR = Path("/content/AXIOM_DE-RD")

# Chỉnh nếu dữ liệu nằm ở vị trí khác.
DATA_ROOT = Path("/content/drive/MyDrive/AXIOM_DE-RD/data")
DATASET_DIR = Path(
    "/content/drive/MyDrive/iSE_DE/vidore_v3/vidore_v3_pharmaceuticals"
)
assert DATASET_DIR.is_dir(), f"Không tìm thấy dataset: {DATASET_DIR}"

RUN_NAME = "vidore-v3-pharmaceuticals-chandra2"

openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
assert openrouter_api_key, "Thiếu OPENROUTER_API_KEY trong Colab Secrets"
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

source_config = (
    REPO_DIR / "configs/pipeline.vidore-v3-chandra2.yaml"
)
config = json.loads(source_config.read_text(encoding="utf-8"))

config["local_input"]["path"] = str(DATASET_DIR)
config["local_input"]["recursive"] = True
config["local_input"]["include_extensions"] = [".pdf"]

config["enabled_modules"] = [
    "ingestion",
    "cleaning",
    "enrichment",
    "chunking_embedding",
    "integration",
    "artifacts",
]

# Giữ từng stage trong data/<stage>/benchmarks/...
config["ingested_dir"] = str(
    DATA_ROOT / f"ingested/benchmarks/{RUN_NAME}"
)
config["cleaned_dir"] = str(
    DATA_ROOT / f"cleaned/benchmarks/{RUN_NAME}"
)
config["enriched_dir"] = str(
    DATA_ROOT / f"enriched/benchmarks/{RUN_NAME}"
)
config["embedded_dir"] = str(
    DATA_ROOT / f"embedded/benchmarks/{RUN_NAME}"
)
config["output_dir"] = str(
    DATA_ROOT / f"output/benchmarks/{RUN_NAME}"
)
config["parsing"]["chandra2"]["output_dir"] = str(
    DATA_ROOT / f"work/benchmarks/{RUN_NAME}"
)

config["chunking_embedding"] = {
    "chunker": "fixed_overlap",
    "chunker_params": {
        "n_words": 512,
        "overlap": 128,
    },
    "max_rows_per_chunk": 20,
    "embedder": "openrouter_te3s",
    "embedder_params": {
        "model": "openai/text-embedding-3-small",
        "dimension": 1536,
        "api_key_env": "OPENROUTER_API_KEY",
        "batch_size": 64,
        "cache_dir": str(
            DATA_ROOT / "work/embedding_cache/text-embedding-3-small"
        ),
        "app_title": "AXIOM_DE-RD",
    },
    "retrieval_profile": "hybrid_default",
}


RUNTIME_CONFIG = Path(
    "/content/pipeline.vidore-v3-pharmaceuticals-full.json"
)
RUNTIME_CONFIG.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)
print("Runtime config:", RUNTIME_CONFIG)
print("Dataset:", config["local_input"]["path"])
print("Modules:", config["enabled_modules"])
print("Output:", config["output_dir"])
print(
    "Embedding model:",
    config["chunking_embedding"]["embedder_params"]["model"],
)
# print("PDF count:", pdf_count)
print("Runtime config:", RUNTIME_CONFIG)
print("Output directory:", config["output_dir"])
print("Enabled modules:", config["enabled_modules"])
print("Embedding model:", config["chunking_embedding"]["embedder_params"]["model"])

In [ ]:
import json
from pathlib import Path

# ============================================================
# Patch Chandra runtime config
# ============================================================

RUNTIME_CONFIG = Path(
    "/content/pipeline.vidore-v3-pharmaceuticals-full.json"
)

config = json.loads(
    RUNTIME_CONFIG.read_text(encoding="utf-8")
)

chandra_config = config["parsing"]["chandra2"]

chandra_config["continuous_page_queue"] = True

# GPU-dependent request concurrency
chandra_config["max_workers"] = CHANDRA_MAX_WORKERS

# CPU-dependent PDF rendering concurrency
chandra_config["render_processes"] = (
    CHANDRA_RENDER_PROCESSES
)

# Keep one page per HTTP request
chandra_config["request_batch_size"] = 1


RUNTIME_CONFIG.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)


print("\nApplied runtime config:")
print({
    "continuous_page_queue":
        chandra_config["continuous_page_queue"],

    "max_workers":
        chandra_config["max_workers"],

    "request_batch_size":
        chandra_config["request_batch_size"],

    "render_processes":
        chandra_config["render_processes"],
})

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/AXIOM_DE-RD")
RUNTIME_CONFIG = Path(
    "/content/pipeline.vidore-v3-pharmaceuticals-full.json"
)
LOG_PATH = Path("/content/vidore-v3-pharmaceuticals.log")

log_handle = LOG_PATH.open("w", encoding="utf-8")

pipeline_process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "scripts/run_pipeline.py",
        "--config",
        str(RUNTIME_CONFIG),
    ],
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
)

print("Pipeline PID:", pipeline_process.pid)
print("Log:", LOG_PATH)

In [ ]:
# pipeline_process.terminate()
# pipeline_process.wait(timeout=30)
# log_handle.close()

# print("Pipeline đã dừng")

In [ ]:
from pathlib import Path

INGESTED_ROOT = Path(
    "/content/drive/MyDrive/AXIOM_DE-RD/data/"
    "ingested/benchmarks/vidore-v3-pharmaceuticals-chandra2"
)

runs = sorted(
    (path for path in INGESTED_ROOT.iterdir() if path.is_dir()),
    key=lambda path: path.stat().st_mtime,
)

latest_run = runs[-1]
documents = list((latest_run / "documents").glob("*.json"))

print("Run:", latest_run.name)
print("Persisted documents:", len(documents), "/ 52")
print("Metadata exists:", (latest_run / "metadata.json").is_file())